# Two things about this pipeline that cost me submissions

Both of these are about the public dual-seed / ILP / safe-division lineage that most of the
0.93–0.94 band is running. Neither is a configuration and there is nothing here to fork for score.
They are the two facts I wish I had known a week earlier.

1. **The PROXY_SCORE that every kernel on this lineage prints ranks three of my own submissions
   in exactly the reverse of their leaderboard order.** It is not a weak signal on those three,
   it is an inverted one, and the thing that predicts the direction is the predicted node count.
2. **The ILP cannot produce a division at the `BIOHUB_ILP_DIVISION_WEIGHT` this lineage ships.**
   Not "rarely does" — cannot, for arithmetic reasons, at any value ≥ 1. Every division in these
   submissions comes from the safe-division post-process. Section 2 proves it in six lines you
   can run.


## 1. The proxy is inverted on node-count changes

The validator built into these kernels scores four held-out training movies with the official
formula and prints one number. Three of this account's submissions, all from the same lineage,
all scored on the real leaderboard:

| submission | held-out PROXY_SCORE | public LB | nodes written | forks |
|---|---:|---:|---:|---:|
| detection threshold raised to 0.985 | **0.9434** | **0.937** | 115,783 | 149 |
| the published 0.941 configuration | 0.9418 | 0.941 | 119,279 | 94 |
| that, plus one safe-division knob | 0.9414 | **0.942** | 119,315 | 87 |

Read the first two columns against each other. The proxy puts them in the order
0.9434 > 0.9418 > 0.9414. The leaderboard puts them in the order 0.937 < 0.941 < 0.942.
**The proxy ranking is the exact reverse of the leaderboard ranking on these three.**

Now read the proxy against the node count instead. The run with the highest proxy wrote 3,532
fewer nodes than the run with the highest leaderboard score, and it got there by raising the
detection threshold, which does nothing but delete candidate nodes.

The metric is `adjusted_edge_jaccard + 0.1 · division_jaccard`, and the adjustment in the first
term is against the number of nodes you predict, because the ground truth is sparse — only a
small fraction of predicted nodes have a labelled counterpart at all. Delete nodes and the
adjustment moves in your favour on the four movies you can see. On the test set it does not.

**The rule I now use:** the proxy is admissible only for comparing two candidates whose node
count is essentially unchanged — a pure re-linking, where only edges move. For anything that adds
or removes nodes, it is not a weak gate, it is a misleading one, and the only honest gate is a
structural check plus the leaderboard.

This is the same shape of finding that
[busyaprime](https://www.kaggle.com/code/busyaprime/biohub-0-942-lb-one-knob-past-the-public-line)
reported from the other direction: their higher-scoring detection threshold also printed a
*lower* proxy than the published one. Two accounts, opposite directions on the same knob, same
conclusion about the instrument.


In [ ]:
rows = [
    ("det threshold 0.985",            0.9434, 0.937, 115783, 149),
    ("published 0.941 configuration",  0.9418, 0.941, 119279,  94),
    ("+ one safe-division knob",       0.9414, 0.942, 119315,  87),
]
print(f"{'submission':<32}{'proxy':>9}{'LB':>8}{'nodes':>10}{'forks':>7}")
for name, proxy, lb, nodes, forks in rows:
    print(f"{name:<32}{proxy:>9.4f}{lb:>8.3f}{nodes:>10,}{forks:>7}")

def spearman(a, b):
    ra = sorted(range(len(a)), key=lambda i: a[i])
    rb = sorted(range(len(b)), key=lambda i: b[i])
    ra = [ra.index(i) for i in range(len(a))]
    rb = [rb.index(i) for i in range(len(b))]
    n = len(a)
    d2 = sum((x - y) ** 2 for x, y in zip(ra, rb))
    return 1 - 6 * d2 / (n * (n * n - 1))

proxy = [r[1] for r in rows]; lb = [r[2] for r in rows]; nodes = [r[3] for r in rows]
print(f"\nSpearman(proxy, LB)  = {spearman(proxy, lb):+.2f}")
print(f"Spearman(nodes, LB)  = {spearman(nodes, lb):+.2f}")
print("\nThree points is three points. It is enough to stop trusting the proxy,")
print("not enough to trust the node count as a predictor in its place.")


## 2. The ILP cannot divide at the weight this lineage ships

This is the outgoing-flow constraint, verbatim from `tracksdata/solvers/_ilp_solver.py`:

```py
self._constraints.add(
    self._disappear_vars[source_id] + sum(self._edge_vars[edge_id] for edge_id in edge_ids)
    == self._node_vars[source_id] + self._division_vars[source_id]
)
```

A node that is kept and does not disappear has `node_vars = 1`, `disappear_vars = 0`. So the
number of outgoing edges equals `1 + division_vars`, which means **out-degree 2 if and only if
the division variable is 1**. There is no other way for the solver to fork.

The objective is minimised. Giving a parent a second daughter with edge probability `p₂` changes
it by

```
Δ  =  (edge coefficient for the new edge)  +  division_weight  −  appearance_weight
   =  −p₂  +  division_weight  −  appearance_weight
```

The `appearance_weight` term is there because if that daughter is not linked to this parent it
has to start a new track instead, and that costs `appearance_weight`.

This lineage sets `BIOHUB_ILP_APPEARANCE_WEIGHT = 0.0`, so

```
Δ  =  division_weight − p₂
```

and the solver forks whenever **`p₂ ≥ division_weight`** (at exact equality Δ is zero, both
solutions are optimal, and which one you get is the solver's tie-break). The division weight is
not a penalty that trades off against evidence — it *is* the probability threshold, in the same
units.

Since `p₂ ≤ 1` always, **any `division_weight ≥ 1` makes a fork impossible.** The published
configurations on this lineage ship `1.2`.


In [ ]:
import numpy as np
from scipy.optimize import milp, LinearConstraint, Bounds

def fork_or_not(p1, p2, division_weight, appearance_weight=0.0):
    """The constraint above, written out for one parent and two candidate daughters.

    Variables: [edge_1, edge_2, division, disappearance], all binary, node fixed to 1.
    Constraint: disappearance + edge_1 + edge_2 == node + division, with node = 1.
    Objective (minimised): -p1*edge_1 - p2*edge_2 + division_weight*division
                           + appearance_weight*(daughters not linked)
    """
    c = np.array([-p1, -p2, division_weight, 0.0])
    c[0] -= appearance_weight          # linking a daughter saves its appearance cost
    c[1] -= appearance_weight
    constraint = LinearConstraint(np.array([[1.0, 1.0, -1.0, 1.0]]), 1, 1)
    res = milp(c=c, constraints=constraint, integrality=np.ones(4), bounds=Bounds(0, 1))
    edge_1, edge_2, division, _ = np.round(res.x).astype(int)
    return edge_1 + edge_2, division

P2 = 0.90
print(f"second daughter has edge probability p2 = {P2}, appearance_weight = 0.0\n")
for w in (1.2, 1.0, 0.9, 0.8, 0.5):
    kept, div = fork_or_not(0.95, P2, w)
    print(f"  division_weight = {w:<4} -> {kept} edge(s) kept, division={div}"
          f"   {'DIVISION' if div else 'no division'}")


Nothing is installed above: `scipy.optimize.milp` is already in the image, and the model is
the constraint quoted from `_ilp_solver.py` written out for three nodes. If `tracksdata` happens to
be importable in your session the next cell runs the same thing through the real solver — but do
not `pip install` it here, that pulls a numpy that the image cannot import afterwards.


In [ ]:
try:
    import polars as pl, tracksdata as td
    from tracksdata.solvers import ILPSolver
    from tracksdata.constants import DEFAULT_ATTR_KEYS as K
except Exception as exc:
    print("tracksdata not importable here, skipping:", type(exc).__name__)
else:
    def graph(p1, p2):
        g = td.graph.InMemoryGraph()
        g.add_edge_attr_key("w", pl.Float64, 0.0)
        parent, d1, d2 = g.add_node({"t": 0}), g.add_node({"t": 1}), g.add_node({"t": 1})
        g.add_edge(parent, d1, {"w": -p1})       # edge coefficient is -probability
        g.add_edge(parent, d2, {"w": -p2})
        return g

    for w in (1.2, 1.0, 0.9, 0.8):
        solved = ILPSolver(edge_weight="w", appearance_weight=0.0,
                           disappearance_weight=0.0, division_weight=w).solve(graph(0.95, 0.90))
        kept = sum(1 for v in solved.edge_attrs()[K.SOLUTION] if v)
        print(f"  tracksdata: division_weight = {w:<4} -> {kept} edge(s) kept")


The fork appears below 0.90 and at nothing above it, with the crossing landing exactly on
`p₂ = division_weight`. Change `P2` and the crossing moves with it.

The row at `division_weight = 0.90` is worth a second look, because it is the tie. There
Δ is exactly zero, both solutions are optimal, and the answer is whichever one the solver
happens to return: scipy's HiGHS returns no division there, and when I ran the same case through
tracksdata's SCIP it returned the division. That disagreement is not a bug in either — it is what
a tie looks like, and it is the cleanest confirmation that the threshold really is at
`p₂ = division_weight` and not somewhere near it.

**Two consequences worth acting on.**

*Sweeping `division_weight` above 1.0 is a no-op.* Every value ≥ 1 gives the same solution — the
one with no ILP divisions at all. If you tried 1.0, 2.0 and 3.0 and got the same score three
times, that is not a plateau, that is the same run.

*Every division in these submissions comes from the post-process, not the solver.* The safe-division
rule and the DeepCenter veto are not refinements on top of what the ILP found; on this
configuration they are the entire division channel. That is worth knowing before you spend a
submission tuning the solver's division weight, and it is why the safe-division distance caps
have been where the movement is.

I am not claiming the interval below 1.0 is where you should be — I have not established that a
solver-side division channel beats the post-process one, and the appearance weight, which nobody
on this lineage has moved off 0.0, sits in the same expression and would shift the threshold.
I am claiming that the interval above 1.0 contains no information, and that a lot of sweeps have
been run there.

---

Credit to [pilkwang](https://www.kaggle.com/code/pilkwang/biohub-cell-tracking-two-seeds-logit-blend),
whose weights and pipeline this whole lineage runs on, and to
[busyaprime](https://www.kaggle.com/code/busyaprime/biohub-0-941-lb-every-knob-and-who-set-it),
whose provenance tables are how I found out which knobs had never been moved. The solver is
[tracksdata](https://github.com/royerlab/tracksdata) by the Royer lab.
